# Employee Churn: Exploratory Data Analysis

## 0. Data loading and initial inspection
Check dataset size, missing values and duplicates. `left`: 0 = Stayed, 1 = Left. Run this notebook from the repository root.

Exact duplicate rows are removed, although identical records may represent different employees. Results before and after removal are shown below.

This overview uses the full HR dataset; Sections 1–5 use only `train.csv`. Findings show associations, not causes.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display
from matplotlib.ticker import PercentFormatter
from matplotlib.patches import Patch

# Keep the same colors, fonts and subtle grids throughout the notebook.
sns.set_theme(style="white", context="notebook", rc={
    "figure.facecolor": "white", "axes.facecolor": "white",
    "figure.dpi": 120, "axes.axisbelow": True,
    "axes.edgecolor": "#CBD2D9", "axes.linewidth": 0.8,
    "axes.labelcolor": "#374151", "text.color": "#1F2937",
    "xtick.color": "#4B5563", "ytick.color": "#4B5563",
    "grid.color": "#E9EDF2", "grid.linewidth": 0.6,
    "axes.titleweight": "semibold", "axes.titlesize": 13,
    "axes.titlelocation": "left", "axes.titlepad": 16,
    "axes.labelsize": 11, "xtick.labelsize": 10, "ytick.labelsize": 10,
    "legend.frameon": False, "patch.linewidth": 0,
})
STATUS_ORDER = ["Stayed", "Left"]
STATUS_COLORS = {"Stayed": "#4C78A8", "Left": "#D97757"}
TOPIC_COLORS = {
    "projects": "#2A9D8F",
    "tenure": "#8172B2",
    "salary": "#B58A35",
    "department": "#607D9C",
}

def style_axes(ax, grid_axis="y"):
    ax.set_axisbelow(True)
    ax.grid(False, which="both")
    ax.grid(True, axis=grid_axis, color="#E9EDF2", linewidth=0.6)
    ax.tick_params(axis="both", length=0, pad=7)
    sns.despine(ax=ax)

In [ ]:
raw_data = pd.read_csv('data/HR_Dataset.csv')
raw_data.columns = raw_data.columns.str.strip()

print('Dataset shape:', raw_data.shape)
raw_data.info()

### Column types and observed values (full dataset overview)
Unique categories, observed numeric ranges and 0/1 meanings from `HR_Dataset.csv`.

In [ ]:
# Describe each column using the values observed in the full dataset.
binary_meanings = {
    'Work_accident': '0 = No recorded accident; 1 = Recorded accident',
    'left': '0 = Stayed; 1 = Left',
    'promotion_last_5years': '0 = Not promoted; 1 = Promoted',
}
column_rows = []
for column in raw_data.columns:
    values = raw_data[column]
    if column in binary_meanings:
        column_type = 'Binary'
        description = '0, 1 (' + binary_meanings[column] + ')'
    elif values.dtype == 'object':
        column_type = 'Categorical'
        description = ', '.join(sorted(values.dropna().unique()))
    else:
        column_type = 'Numeric'
        description = f'{values.min():g} – {values.max():g}'

    column_rows.append({
        'Column': column,
        'Type': column_type,
        'Data type': str(values.dtype),
        'Values / range': description,
        'Missing values': values.isna().sum(),
    })

column_values = pd.DataFrame(column_rows)
# Formatting only: left-align text and wrap long category lists.
display(
    column_values.style.hide(axis='index')
    .set_properties(**{'text-align': 'left', 'vertical-align': 'top'})
    .set_properties(subset=['Values / range'], **{
        'white-space': 'normal', 'overflow-wrap': 'anywhere', 'max-width': '700px',
    })
)

In [ ]:
clean_data = raw_data.drop_duplicates().reset_index(drop=True)
clean_data["Status"] = clean_data["left"].map({0: "Stayed", 1: "Left"})
clean_churn_rate = clean_data["left"].mean() * 100
status_counts = clean_data["Status"].value_counts().reindex(STATUS_ORDER)

overview = pd.DataFrame({
    "Dataset": ["Before duplicate removal", "After duplicate removal (analysis)"],
    "Employees (n)": [len(raw_data), len(clean_data)],
    "Employees who left": [int(raw_data["left"].sum()), int(clean_data["left"].sum())],
    "Churn rate (%)": [raw_data["left"].mean() * 100, clean_churn_rate],
})
display(overview.style.format({"Churn rate (%)": "{:.2f}"}).hide(axis="index"))
print(f"Exact duplicate rows removed: {raw_data.duplicated().sum():,}")
display(raw_data.isna().sum().rename("Missing values").to_frame())

status_summary = status_counts.rename("Employees (n)").to_frame()
status_summary["Share (%)"] = status_summary["Employees (n)"] / len(clean_data) * 100
display(status_summary.style.format({"Share (%)": "{:.2f}"}))

In [ ]:
ax = status_counts.plot.bar(color=[STATUS_COLORS[status] for status in STATUS_ORDER], rot=0, width=0.6, edgecolor="none", figsize=(6, 4))
style_axes(ax)
for bar, count in zip(ax.patches, status_counts):
    ax.annotate(f"{count:,} ({count / len(clean_data):.1%})", (bar.get_x() + bar.get_width() / 2, bar.get_height()), xytext=(0, 5), textcoords="offset points", ha="center")
ax.set(title="Employee Churn Overview", xlabel="Employment status", ylabel="Employees (count)")
ax.set_ylim(0, status_counts.max() * 1.18)
plt.tight_layout()
plt.show()

In [ ]:
display(clean_data.head())
display(clean_data.describe(include='all'))

**Key takeaway:** The dataset contains 14,999 records, no missing values and 3,008 exact duplicates. Removing duplicates changes the churn rate from 23.8% to 16.6%, making the duplicate-handling assumption important.

## Training data analysis
All analyses below use only `data/train.csv`.

In [ ]:
# All feature analysis below uses this training split only.
train = pd.read_csv('data/train.csv')
train.columns = train.columns.str.strip()
train["Status"] = train["left"].map({0: "Stayed", 1: "Left"})
training_rate = train["left"].mean() * 100
status_counts = train["Status"].value_counts().reindex(STATUS_ORDER, fill_value=0)
print(f"Analysis dataset: data/train.csv | Employees: {len(train):,} | Churn rate: {training_rate:.2f}%")
training_summary = status_counts.rename("Employees (n)").to_frame()
training_summary["Share (%)"] = training_summary["Employees (n)"] / len(train) * 100
display(training_summary.style.format({"Share (%)": "{:.2f}"}))

## 1. Satisfaction and evaluation
Compare satisfaction and evaluation scores between employees who stayed and left. Each group's histogram sums to 100%; bar heights show the share within that group, not churn rates.

In [ ]:
score_columns = {"satisfaction_level": "Satisfaction level", "last_evaluation": "Last evaluation"}
score_bins = np.linspace(0, 1, 26)  # 25 common bins for both statuses
fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharex=True)
for ax, (column, label) in zip(axes, score_columns.items()):
    sns.histplot(data=train, x=column, hue="Status", hue_order=STATUS_ORDER,
                 bins=score_bins, stat="percent", common_norm=False,
                 multiple="layer", palette=STATUS_COLORS, alpha=0.6,
                 edgecolor="white", linewidth=0.4, legend=False, ax=ax)
    style_axes(ax)
    ax.set(title=label, xlabel=label, ylabel="Employees within each status (%)", xlim=(0, 1))
    ax.yaxis.set_major_formatter(PercentFormatter(xmax=100))
legend_handles = []
for status in STATUS_ORDER:
    legend_handles.append(Patch(facecolor=STATUS_COLORS[status], label=status, alpha=0.6))
fig.legend(handles=legend_handles,
           loc="upper center", bbox_to_anchor=(0.5, 1.0), ncol=2)
fig.text(0.5, 0.015, f"Training split | Stayed: n={status_counts['Stayed']:,} | Left: n={status_counts['Left']:,} | Each status sums to 100%",
         ha="center", fontsize=10, color="#6B7280")
fig.tight_layout(rect=(0, 0.06, 1, 0.91))
plt.show()

In [ ]:
display(train.groupby("Status")[list(score_columns)].agg(["count", "mean", "median"]).reindex(STATUS_ORDER).round(3))

**Key takeaway:** Leavers have clearly lower satisfaction (median 0.41 vs 0.69), with distinct clusters near 0.1 and 0.4. Their evaluation scores are not simply lower or higher: means are nearly identical (0.716 vs 0.723), but leavers split into a low-evaluation group (~0.45–0.58) and a high-evaluation group (~0.8–1.0).

## 2. Workload
Compare churn rates by number of projects and monthly working hours between employees who stayed and left. The dashed line marks the training-set churn rate; tables show group sizes and median hours.

In [ ]:
# A small helper avoids repeating the same group calculation in each section.
def churn_summary(column):
    summary = train.groupby(column, observed=True)['left'].agg(['size', 'sum', 'mean'])
    summary.columns = ['Total employees', 'Employees who left', 'Churn rate (%)']
    summary['Churn rate (%)'] = summary['Churn rate (%)'] * 100
    return summary

project_summary = churn_summary('number_project')
display(project_summary.style.format({'Churn rate (%)': '{:.2f}'}))

hours_summary = train.groupby('Status')['average_montly_hours'].agg(['size', 'median'])
hours_summary.columns = ['Total employees', 'Median monthly hours']
hours_summary = hours_summary.reindex(STATUS_ORDER)
hours_summary.index.name = 'Employment status'
display(hours_summary.style.format({'Total employees': '{:,.0f}', 'Median monthly hours': '{:.0f}'}))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].bar(project_summary.index.astype(str), project_summary['Churn rate (%)'],
          color=TOPIC_COLORS['projects'], width=0.65, edgecolor='none', zorder=3)
axes[0].axhline(training_rate, color='#4B5563', linestyle=(0, (4, 4)), linewidth=1, zorder=4)
axes[0].text(0.99, 0.97, f'Average Training-set churn rate: {training_rate:.1f}%',
           transform=axes[0].transAxes, ha='right', va='top', fontsize=9, color='#4B5563')
rate_limit = max(30, project_summary['Churn rate (%)'].max() * 1.25, training_rate * 1.25)
axes[0].set(title='Churn Rate by Number of Projects', xlabel='Number of projects', ylabel='Churn rate (%)', ylim=(0, rate_limit))
axes[0].set_yticks(np.arange(0, min(rate_limit, 100) + 0.01, 20))
axes[0].yaxis.set_major_formatter(PercentFormatter(xmax=100))
for position, rate in enumerate(project_summary['Churn rate (%)']):
    axes[0].annotate(f'{rate:.1f}%', (position, rate), xytext=(0, 5),
                   textcoords='offset points', ha='center', fontsize=9)
style_axes(axes[0])
sns.boxplot(data=train, x="Status", y="average_montly_hours", order=STATUS_ORDER,
            hue="Status", hue_order=STATUS_ORDER, palette=STATUS_COLORS, dodge=False,
            width=0.5, linewidth=1, saturation=1,
            medianprops={"color": "#1F2937", "linewidth": 1.8},
            flierprops={"marker": "o", "markersize": 3, "alpha": 0.3}, ax=axes[1])
style_axes(axes[1])
hour_medians = train.groupby("Status")["average_montly_hours"].median()
axes[1].set_xticks(range(2), [f"{status}\n(median={hour_medians[status]:.0f} h)" for status in STATUS_ORDER])
axes[1].set(title="Monthly Working Hours: Left vs Stayed", xlabel="Employment status", ylabel="Working hours per month")
plt.tight_layout()
plt.show()

**Key takeaway:** Churn has a U-shaped relationship with number of projects: lowest at 3 (1.1%) and 4 (6.4%), then rising at 5 (15.7%), 6 (43.4%) and 7 (100%, n=100), but also high at 2 (54.4%). Leavers also have a wider spread of monthly hours (IQR ~146–264 vs ~162–238), suggesting both under- and over-worked employees leave.

## 3. Tenure and promotion
Compare churn rates by years at the company and promotion in the last five years. Tables show group sizes and the number who left.

In [ ]:
tenure_summary = churn_summary('time_spend_company')
display(tenure_summary.style.format({'Churn rate (%)': '{:.2f}'}))

promotion_summary = churn_summary('promotion_last_5years')
promotion_summary = promotion_summary.rename(index={0: 'Not promoted', 1: 'Promoted'})
promotion_summary.index.name = 'Promotion in the last five years'
display(promotion_summary.style.format({'Churn rate (%)': '{:.2f}'}))

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(tenure_summary.index.astype(str), tenure_summary['Churn rate (%)'],
          color=TOPIC_COLORS['tenure'], width=0.65, edgecolor='none', zorder=3)
ax.axhline(training_rate, color='#4B5563', linestyle=(0, (4, 4)), linewidth=1, zorder=4)
ax.text(0.99, 0.97, f'Average Training-set churn rate: {training_rate:.1f}%',
           transform=ax.transAxes, ha='right', va='top', fontsize=9, color='#4B5563')
rate_limit = max(30, tenure_summary['Churn rate (%)'].max() * 1.25, training_rate * 1.25)
ax.set(title='Churn Rate by Tenure', xlabel='Years at company', ylabel='Churn rate (%)', ylim=(0, rate_limit))
ax.set_yticks(np.arange(0, min(rate_limit, 100) + 0.01, 10))
ax.yaxis.set_major_formatter(PercentFormatter(xmax=100))
for position, rate in enumerate(tenure_summary['Churn rate (%)']):
    ax.annotate(f'{rate:.1f}%', (position, rate), xytext=(0, 5),
                   textcoords='offset points', ha='center', fontsize=9)
style_axes(ax)
plt.tight_layout()
plt.show()

**Key takeaway:** Churn is concentrated in years 3-6 and peaks at year 5 (45.5%); it is near zero at year 2 (1.0%) and zero beyond year 6 (0 of 190 employees). Promoted employees churn less (5.1% vs 16.8%), but this group is small (n=136, only 7 leavers), so the estimate should be interpreted with caution. Promotion may also be confounded with tenure and salary.

## 4. Salary, department and work accidents
Compare churn rates across salary levels, departments and recorded work-accident groups. Departments are ranked by churn rate; tables show group sizes and the number who left.

In [ ]:
salary_summary = churn_summary('salary').reindex(['low', 'medium', 'high'])
department_summary = churn_summary('Departments').sort_values('Churn rate (%)', ascending=False)
display(salary_summary.style.format({'Churn rate (%)': '{:.2f}'}))
display(department_summary.style.format({'Churn rate (%)': '{:.2f}'}))

accident_summary = churn_summary('Work_accident')
accident_summary = accident_summary.rename(index={0: 'No recorded accident', 1: 'Recorded accident'})
accident_summary.index.name = 'Recorded work accident'
display(accident_summary.style.format({'Churn rate (%)': '{:.2f}'}))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 7))
axes[0].bar(salary_summary.index.astype(str), salary_summary['Churn rate (%)'],
          color=TOPIC_COLORS['salary'], width=0.65, edgecolor='none', zorder=3)
axes[0].axhline(training_rate, color='#4B5563', linestyle=(0, (4, 4)), linewidth=1, zorder=4)
axes[0].text(0.99, 0.97, f'Average Training-set churn rate: {training_rate:.1f}%',
           transform=axes[0].transAxes, ha='right', va='top', fontsize=9, color='#4B5563')
rate_limit = max(30, salary_summary['Churn rate (%)'].max() * 1.25, training_rate * 1.25)
axes[0].set(title='Churn Rate by Salary Level', xlabel='Salary level', ylabel='Churn rate (%)', ylim=(0, rate_limit))
axes[0].set_yticks(np.arange(0, min(rate_limit, 100) + 0.01, 5))
axes[0].yaxis.set_major_formatter(PercentFormatter(xmax=100))
for position, rate in enumerate(salary_summary['Churn rate (%)']):
    axes[0].annotate(f'{rate:.1f}%', (position, rate), xytext=(0, 5),
                   textcoords='offset points', ha='center', fontsize=9)
style_axes(axes[0])

# Departments use horizontal bars so their names are easy to read.
axes[1].barh(department_summary.index, department_summary['Churn rate (%)'],
             color=TOPIC_COLORS['department'], height=0.65, edgecolor='none', zorder=3)
axes[1].invert_yaxis()
axes[1].axvline(training_rate, color='#4B5563', linestyle=(0, (4, 4)), linewidth=1, zorder=4)
axes[1].text(training_rate, 1.015, f'Average Training-set churn rate: {training_rate:.1f}%',
             transform=axes[1].get_xaxis_transform(), ha='center', va='bottom',
             fontsize=9, color='#4B5563', clip_on=False)
rate_limit = max(30, department_summary['Churn rate (%)'].max() * 1.25, training_rate * 1.25)
axes[1].set(title='Churn Rate by Department', xlabel='Churn rate (%)',
            ylabel='Department', xlim=(0, rate_limit * 1.1))
axes[1].set_xticks(np.arange(0, min(rate_limit, 100) + 0.01, 5))
axes[1].xaxis.set_major_formatter(PercentFormatter(xmax=100))
for position, rate in enumerate(department_summary['Churn rate (%)']):
    axes[1].annotate(f'{rate:.1f}%', (rate, position), xytext=(5, 0),
                     textcoords='offset points', va='center', fontsize=9)
style_axes(axes[1], grid_axis='x')
plt.tight_layout()
plt.show()

**Key takeaway:** Churn decreases sharply with salary (low 20.4%, medium 14.8%, high 4.1%). Departments differ only modestly: most fall between 15% and 19%, with R&D (11.5%) and management (10.9%) lowest, and these groups are small. Employees with a recorded work accident have lower churn (5.7% vs 18.6%), which is counterintuitive and may reflect confounding rather than any protective effect.

## Overall takeaway for classification
The training-data EDA suggests that employee churn is associated with several features rather than a single factor. Satisfaction, workload, tenure and salary show clear differences in churn patterns. Evaluation scores should also be retained, since employees who leave do not necessarily have low evaluations.